# Notebook 06: Fine-tuning & Alignment — Concepts Overview

**Time:** 20 minutes  
**Prerequisites:** Notebook 05 complete  
**Goal:** Understand what happens *after* pretraining — SFT, LoRA, DPO, and PPO

> **📌 Note:** Full hands-on implementation of fine-tuning is covered in a dedicated later class.
> This notebook builds **conceptual understanding** using Claude as your interactive guide.

## The Three-Stage Lifecycle

```
Stage 1: Pretraining         → Base model (predicts next token, no "personality")
    ↓
Stage 2: Supervised Fine-Tuning (SFT)  → Instruction-following model
    ↓  
Stage 3: Alignment (RLHF / DPO / PPO) → Helpful, harmless, honest model
```

You just built the data pipeline for **Stage 1** (notebooks 04–05). This notebook explains what happens in stages 2 and 3.

In [1]:
import os, sys, time
from pathlib import Path

notebook_dir = os.getcwd()
parent_dir   = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'))

from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import format_response, append_to_reflection
import src.config as config

client  = LLMClient(path=config.PATH)
tracker = CostTracker()

outputs_dir = os.path.join('..', 'outputs')
os.makedirs(outputs_dir, exist_ok=True)

print("✅ Setup complete — ready for Notebook 06")

/Users/scottlai/Documents/inferenceAI/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✓ Claude API client initialized
  Default model: claude-sonnet-4-6
  Available: claude-sonnet-4-6, claude-opus-4-6, claude-haiku-4-5-20251001
✅ Setup complete — ready for Notebook 06


---

## Part 1: The Effect of SFT — Base vs Instruction-Tuned Models

A **base model** is trained purely to predict the next token. It will complete text in any way that's statistically consistent with its training data — which might not be what you want.

An **instruction-tuned** model (after SFT) has learned to respond helpfully to questions and commands.

In [2]:
print("=" * 65)
print("🧪 Experiment 1: Base Model vs Instruction-Tuned Model")
print("=" * 65)
print()

# Same prompt given to both a base-style and instruction-style prompt
neutral_prompt = "Explain the transformer architecture:"

# Base model style: just continue the text (complete mode)
base_style_system = None   # no system prompt = closer to base model behavior

# Instruction-tuned style: structured helpful response
instruction_style_system = (
    "You are a helpful ML engineering tutor. When asked to explain a concept, "
    "provide a clear, structured answer with: (1) one-sentence definition, "
    "(2) three key components, (3) one practical implication."
)

print(f"Prompt: \"{neutral_prompt}\"")
print()

print("--- Style A: No system prompt (base model behavior) ---")
resp_base = client.generate(
    prompt=neutral_prompt, system=None,
    max_tokens=200, temperature=1.0
)
if "error" not in resp_base:
    tracker.add_call(resp_base)
    print(resp_base['content'])

print()
print("--- Style B: With instruction-tuning system prompt ---")
resp_instr = client.generate(
    prompt=neutral_prompt, system=instruction_style_system,
    max_tokens=300, temperature=0.5
)
if "error" not in resp_instr:
    tracker.add_call(resp_instr)
    print(resp_instr['content'])

print()
print("💡 SFT training encodes the 'instruction-style' behavior into model weights,")
print("   so you don't need the system prompt to trigger it. The model just 'knows'.")

🧪 Experiment 1: Base Model vs Instruction-Tuned Model

Prompt: "Explain the transformer architecture:"

--- Style A: No system prompt (base model behavior) ---
# Transformer Architecture

## Overview

The Transformer (introduced in *"Attention Is All You Need"*, Vaswani et al., 2017) replaced recurrent networks with a purely **attention-based** architecture, enabling parallelization and better long-range dependency modeling.

---

## High-Level Structure

```
Input → [Encoder Stack] → Context Representations
                                    ↓
Output → [Decoder Stack] → Final Output
```

---

## Core Components

### 1. Input Embeddings + Positional Encoding
Since there's no recurrence, position must be injected explicitly.

```
Token Embedding + Positional Encoding → Input Representation

PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10

--- Style B: With instruction-tuning system prompt ---
# Transformer Architecture

## One-Sentence Definition
A transforme

---

## Part 2: What SFT Data Looks Like

SFT trains on **instruction-response pairs**. Here's the format used by major labs:

```json
{
  "instruction": "Summarize this paper abstract in one sentence.",
  "input": "We present a novel method for training large language models...",
  "output": "This paper introduces a new LLM training approach that..."
}
```

The quality of SFT data matters enormously — a small set of high-quality pairs beats a large set of mediocre ones.

In [3]:
print("=" * 65)
print("🧪 Experiment 2: Generate Example SFT Training Pairs")
print("=" * 65)
print()

domain = "machine learning research"   # We'll ask Claude to generate pairs for this

sft_prompt = f"""
Generate 3 high-quality SFT (Supervised Fine-Tuning) training pairs for 
a language model assistant specialized in {domain}.

Each pair should follow this JSON format:
{{"instruction": "...", "input": "...", "output": "..."}}

Requirements for high quality:
- Instructions should be specific and actionable
- Outputs should be correct, helpful, and at appropriate length
- Cover different task types (explanation, summarization, comparison, generation)
- Inputs can be empty ("") if the instruction is self-contained

Return exactly 3 JSON objects, one per line.
"""

resp_sft = client.generate(
    prompt=sft_prompt,
    system="You are a data quality expert for LLM training. Generate realistic, high-quality training examples.",
    max_tokens=800,
    temperature=0.7
)

if "error" not in resp_sft:
    tracker.add_call(resp_sft)
    print("Generated SFT pairs:")
    print(resp_sft['content'])
else:
    print(f"❌ Error: {resp_sft['error']}")

🧪 Experiment 2: Generate Example SFT Training Pairs

Generated SFT pairs:
{"instruction": "Explain the intuition behind the attention mechanism in transformers, including why it was a significant improvement over RNNs for sequence modeling tasks.", "input": "", "output": "The attention mechanism in transformers addresses a fundamental bottleneck in recurrent neural networks: the fixed-size context vector.\n\n**The RNN Problem**\nIn sequence-to-sequence RNNs (e.g., encoder-decoder architectures), the encoder compresses an entire input sequence into a single hidden state vector, which the decoder then uses to generate output. This creates two critical issues:\n1. **Information bottleneck**: Long sequences must be compressed into a fixed-dimensional vector, causing information loss.\n2. **Vanishing gradients**: Backpropagation through many timesteps degrades gradient signals, making it hard to learn long-range dependencies.\n\n**The Attention Intuition**\nAttention asks: *\"When generatin

### 🎯 TODO 1: Generate SFT Pairs for Your Domain

In [ ]:
# TODO 1: Generate SFT training pairs for YOUR project domain

my_domain = "[YOUR DOMAIN HERE]"  # e.g., "medical diagnosis", "legal document review", "code debugging"

my_sft_prompt = f"""
Generate 5 high-quality SFT training pairs for an assistant specialized in {my_domain}.
Use the same format: {{"instruction": "...", "input": "...", "output": "..."}}
Make them realistic for what a practitioner in this field would actually ask.
"""

print("=" * 65)
print("🎯 TODO 1: My Domain SFT Pairs")
print("=" * 65)
print(f"Domain: {my_domain}")
print()

resp_todo1 = client.generate(
    prompt=my_sft_prompt,
    max_tokens=1000,
    temperature=0.7
)

if "error" not in resp_todo1:
    tracker.add_call(resp_todo1)
    print(resp_todo1['content'])

todo1_reflection = """
[YOUR REFLECTION HERE]

- What domain did you choose? Why is it interesting for fine-tuning?
- Looking at the generated pairs: what makes a GOOD training example vs. a bad one?
- How many high-quality pairs do you think you'd need to noticeably improve a base model?
  (Hint: GPT-3's InstructGPT used ~13,000 demonstrations)
"""
print()
print(todo1_reflection)

🎯 TODO 1: My Domain SFT Pairs
Domain: medical diagnosis

Here are 5 high-quality SFT training pairs for a medical diagnosis assistant:

```json
[
  {
    "instruction": "Based on the provided patient presentation, generate a prioritized differential diagnosis with supporting and opposing features for each diagnosis.",
    "input": "45-year-old male presents with 3-week history of progressive exertional dyspnea, orthopnea (2-pillow), and bilateral ankle edema. PMH: HTN x 10 years, poorly controlled (avg BP 160/95). Current meds: amlodipine 5mg. On exam: HR 98, BP 158/96, RR 20, SpO2 94% on room air. JVD present at 45 degrees. Bibasilar crackles. S3 gallop. 2+ pitting edema bilateral lower extremities. CXR shows cardiomegaly and bilateral pleural effusions with Kerley B lines.",
    "output": "## Prioritized Differential Diagnosis\n\n### 1. Hypertensive Heart Disease with Decompensated Heart Failure (HFrEF or HFpEF) — MOST LIKELY\n\n**Supporting features:**\n- 10-year history of poorly c

---

## Part 3: LoRA and Alignment — High-Level Mental Models

### LoRA (Low-Rank Adaptation)

Full fine-tuning updates all model weights — expensive for a 70B model. **LoRA** freezes the base model and trains tiny low-rank adapter matrices:

$$W' = W_0 + \Delta W = W_0 + BA$$

where $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times k}$, with rank $r \ll d$.

**Effect:** A 7B model with r=16 LoRA needs only ~4M trainable parameters (0.06%) instead of 7B!

### DPO vs PPO

Both are alignment methods — they ensure the model prefers helpful, harmless responses over harmful ones.

| | **DPO** (Direct Preference Optimization) | **PPO** (Proximal Policy Optimization) |
|---|---|---|
| **Approach** | Directly optimizes on preference pairs | Reinforcement learning with a reward model |
| **Complexity** | Simple — no reward model needed | Complex — requires training a separate reward model |
| **Stability** | More stable | Can be unstable (reward hacking) |
| **Used by** | Llama 2/3, Mistral, many open models | InstructGPT (GPT-3), GPT-4 (reportedly) |
| **Data needed** | (chosen, rejected) pairs | Human preference scores |

**DPO intuition:** Given a prompt, you have a "chosen" (good) response and a "rejected" (bad) response. DPO directly increases the model's probability of generating the chosen response and decreases the rejected one — no RL needed.

In [6]:
print("=" * 65)
print("🧪 Experiment 3: Ask Claude to Advise on Alignment")
print("=" * 65)
print()

# Use a specific project scenario to make this concrete
alignment_question = f"""
I'm building an AI assistant for {my_domain} that will be used by professionals.
I've collected 10,000 instruction-response pairs (SFT data).
After SFT, I want to align the model to be more helpful and less likely to generate
incorrect domain-specific information.

Please answer:
1. Should I use DPO or PPO? Why?
2. What should my preference pairs (chosen vs. rejected) look like for this domain?
3. How much alignment data do I realistically need?
4. What are the top 2 risks of misalignment in this specific domain?

Be specific and practical.
"""

resp_alignment = client.generate(
    prompt=alignment_question,
    system="You are a senior ML engineer with experience in fine-tuning and aligning language models for production use.",
    max_tokens=600,
    temperature=0.3
)

if "error" not in resp_alignment:
    tracker.add_call(resp_alignment)
    print(format_response(resp_alignment, verbose=True))
else:
    print(f"❌ {resp_alignment['error']}")

🧪 Experiment 3: Ask Claude to Advise on Alignment

Model: claude-sonnet-4-6
Tokens: 173 in, 600 out
Stop reason: max_tokens
# RLHF Alignment Strategy for Medical Diagnosis AI

## 1. DPO vs PPO: Use DPO, But With Eyes Open

### Recommendation: **DPO first, with a path to PPO if needed**

```
DPO Advantages for Your Case:
├── No reward model to maintain (fewer failure points)
├── Stable training (no reward hacking, no KL divergence tuning hell)
├── Works well with smaller datasets (your ~1-3K pairs budget)
├── Easier to audit: preference pairs are human-readable
└── Faster iteration cycles for domain experts to review

PPO Advantages (when you'd switch):
├── Better for nuanced reward shaping (e.g., "be helpful BUT cite sources")
├── Can optimize continuous rewards (confidence calibration scores)
├── Better ceiling performance with large data budgets (10K+ comparisons)
└── Necessary if your reward signal is programmatic (lab value accuracy checks)
```

### The Honest Tradeoff

```python
#

### 🎯 TODO 2: Your Fine-Tuning Strategy Question

In [7]:
# TODO 2: Ask Claude a question about fine-tuning or alignment
#         that is directly relevant to YOUR project.

my_ft_question = """
[YOUR QUESTION HERE]

Examples:
  - "What LoRA rank should I use for adapting qwen3.5:27b to [domain]?"
  - "How do I create DPO preference pairs when there's no clear 'wrong' answer?"
  - "Can I use LoRA to make the model respond in a different language?"
  - "What's the minimum GPU I need to fine-tune a 7B model with QLoRA?"
"""

print("=" * 65)
print("🎯 TODO 2: My Fine-Tuning Question")
print("=" * 65)
print()

resp_todo2 = client.generate(
    prompt=my_ft_question,
    system="You are an expert in LLM fine-tuning. Give a precise, practical answer.",
    max_tokens=500,
    temperature=0.3
)

if "error" not in resp_todo2:
    tracker.add_call(resp_todo2)
    print(format_response(resp_todo2, verbose=False))

todo2_reflection = """
[YOUR REFLECTION HERE]

- What did you ask and what did you learn?
- How does this change your thinking about your project's technical approach?
- Which part of the fine-tuning lifecycle (SFT vs alignment) is most relevant to your project?
"""
print()
print(todo2_reflection)

🎯 TODO 2: My Fine-Tuning Question

It looks like your message contains a placeholder — **"[YOUR QUESTION HERE]"** — rather than an actual question.

Could you replace that with what you'd actually like to know? For example:

- *"What LoRA rank should I use for a 7B model?"*
- *"How do I format data for instruction fine-tuning?"*
- *"What's the difference between SFT and DPO?"*

**Go ahead and ask your real question and I'll give you a precise, practical answer.**


[YOUR REFLECTION HERE]

- What did you ask and what did you learn?
- How does this change your thinking about your project's technical approach?
- Which part of the fine-tuning lifecycle (SFT vs alignment) is most relevant to your project?



## Summary & Reflection

In [8]:
full_reflection = f"""
### Training Lifecycle Understanding

The three stages:
1. Pretraining → base model (I built the data pipeline in notebooks 04-05)
2. SFT → instruction-tuned model (requires {my_domain} instruction-response pairs)
3. Alignment → DPO or PPO to prefer helpful/harmless outputs

### TODO 1 — SFT Data for My Domain ({my_domain})

{todo1_reflection.strip()}

### TODO 2 — Fine-tuning Strategy Question

Question asked: {my_ft_question.strip()[:200]}

{todo2_reflection.strip()}

### Key Takeaways

- LoRA reduces trainable parameters from billions to millions (0.06% of model size)
- DPO is simpler and more stable than PPO for most use cases
- High-quality SFT data (thousands) beats low-quality SFT data (millions)
- Alignment data: (chosen, rejected) pairs where chosen = preferred behavior
"""

reflection_file = append_to_reflection(
    notebook="06",
    section_title="Fine-tuning & Alignment Concepts",
    reflection_content=full_reflection,
    output_dir=os.path.join('..', 'outputs')
)
print(f"✅ Reflection saved: {reflection_file}")
print()
tracker.report()

✅ Reflection saved: ../outputs/homework_reflection.md

💰 API COST REPORT
Total API calls:     7
Total input tokens:  691
Total output tokens: 3,321
Total cost:          $0.0519

Last 5 calls:
  1. [23:18:39] sonnet — 167in/800out — $0.0125
  2. [23:18:54] sonnet — 72in/300out — $0.0047
  3. [23:20:36] sonnet — 69in/1000out — $0.0152
  4. [23:24:45] sonnet — 173in/600out — $0.0095
  5. [23:25:07] sonnet — 136in/121out — $0.0022


## ✅ Notebook 06 Complete!

**What you accomplished:**
- ✅ Understood the base model vs instruction-tuned model difference
- ✅ Generated SFT training examples for your project domain
- ✅ Learned LoRA's parameter-efficiency mechanism
- ✅ Compared DPO vs PPO alignment approaches
- ✅ Got Claude's advice on your specific fine-tuning strategy

**Key resources for deeper study:**
- [LoRA paper](https://arxiv.org/abs/2106.09685) — Hu et al. 2021
- [DPO paper](https://arxiv.org/abs/2305.18290) — Rafailov et al. 2023
- [HuggingFace PEFT docs](https://huggingface.co/docs/peft/) — hands-on LoRA
- [TRL library](https://huggingface.co/docs/trl/) — SFT + DPO training (covered in later class)

**Next:** Open **Notebook 07: Test-Time Scaling** 🧠